In [1]:
# @title ✏️ Settings — the only cell you need to edit
# @markdown **Your model's name** (leave empty for an automatic one, e.g. `RS-Expert-Qwen3.5-2B`)
MODEL_NAME = ""  # @param {type:"string"}
# @markdown **What should it be expert in?**
DOMAIN = "Remote sensing"  # @param ["Remote sensing", "Autonomous driving", "Custom"]
# @markdown Only for **Custom**: describe the field in one sentence, and optionally list sub-topics (comma-separated).
# @markdown Leave the topics empty and the teacher model will propose them.
CUSTOM_DOMAIN = ""  # @param {type:"string"}
CUSTOM_TOPICS = ""  # @param {type:"string"}
# @markdown **Optional:** a Google Drive folder with your own PDFs / .txt / .md notes to teach the model
# @markdown (e.g. `/content/drive/MyDrive/my_papers`). ⚠️ The model can memorise them — don't use confidential or
# @markdown company documents if you publish the model publicly.
DOCS_FOLDER = ""  # @param {type:"string"}
# @markdown **Run size** — `auto` picks the best size for your GPU (≈1–1.5 h on an A100).
# @markdown `quick test` runs everything end-to-end in ≈15 min with a tiny dataset (published as a private test repo).
RUN_SIZE = "auto"  # @param ["auto", "quick test", "large (2x data, slower)"]
# @markdown **Hugging Face upload** — off: nothing is uploaded and no token is needed; your model stays in Google Drive.
UPLOAD_TO_HUGGING_FACE = False  # @param {type:"boolean"}
# @markdown Only used when uploading:
PRIVATE_REPO = False  # @param {type:"boolean"}
MAKE_DEMO_SPACE = True  # @param {type:"boolean"}
SHARE_DATASET = False  # @param {type:"boolean"}
# @markdown **GGUF file** for LM Studio / llama.cpp (saved next to your model)
MAKE_GGUF = True  # @param {type:"boolean"}
# @markdown **Advanced** (leave empty for automatic choices)
TEACHER_MODEL = ""  # @param {type:"string"}
STUDENT_MODEL = ""  # @param {type:"string"}
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
USE_FAST_KERNELS = True  # @param {type:"boolean"}

In [2]:
# @title 1 · Install libraries (≈2 min)
import os, subprocess, sys, tempfile
from importlib.metadata import version as _installed_version
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # less GPU memory fragmentation
import torch

if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU. Menu: Runtime → Change runtime type → pick a GPU (A100 is best) → Save. Then Run all again.")
if "transformers" in sys.modules:
    print("⚠️ Libraries were already imported in this session. If anything fails below: Runtime → Restart session, then Run all.")

_major, _minor = torch.cuda.get_device_capability(0)
# Exact versions this notebook was tested with.
_pkgs = ["transformers==5.19.0", "accelerate==1.15.0", "bitsandbytes==0.50.2", "huggingface_hub==2.2.0", "pypdf==5.9.0"]
# Fast Gated-DeltaNet kernels for Qwen3.5 (pure Triton, no compiling). Needs an Ampere-or-newer GPU (A100, L4, H100…).
if USE_FAST_KERNELS and _major >= 8:
    _pkgs.append("flash-linear-attention==0.5.2")
_constraints = os.path.join(tempfile.gettempdir(), "keep_torch.txt")
with open(_constraints, "w") as f:
    f.write(f"torch=={_installed_version('torch')}\n")  # never replace Colab's torch mid-session
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pkgs, "-c", _constraints], check=True)
print(f"✅ Installed. GPU: {torch.cuda.get_device_name(0)} (compute capability {_major}.{_minor})")

✅ Installed. GPU: NVIDIA A100-SXM4-80GB (compute capability 8.0)


In [3]:
# @title 2 · Engine
# Everything the later steps call lives here: data generation, teacher scoring, distillation training,
# evaluation and publishing. Each step saves its results, so after a disconnect you can just "Run all" again.
import os, re, gc, sys, json, math, time, glob, random, hashlib, shutil, subprocess, contextlib, textwrap
from collections import Counter

import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, GenerationConfig, TextStreamer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ════════════════════════════════════════════════════════════════════════════════════════════════
# Domains
# ════════════════════════════════════════════════════════════════════════════════════════════════
DOMAIN_PRESETS = {
    "Remote sensing": dict(
        short="remote sensing and Earth observation",
        desc=("remote sensing and Earth observation: satellite, aerial and drone imagery (optical, multispectral, "
              "hyperspectral, SAR, LiDAR, thermal), the physics behind it, and the machine-learning and computer-vision "
              "methods used to analyse it"),
        emoji="🛰️",
        tags=["remote-sensing", "earth-observation", "geospatial"],
        topics=[
            "electromagnetic spectrum, radiometry and spectral signatures of land covers",
            "satellite orbits, revisit time, swath width and sensor design",
            "spatial, spectral, radiometric and temporal resolution trade-offs",
            "optical missions: Landsat, Sentinel-2, MODIS, PlanetScope, WorldView",
            "atmospheric correction and surface reflectance (TOA vs BOA)",
            "radiometric calibration and cross-sensor harmonisation",
            "geometric correction, orthorectification and co-registration",
            "map projections, coordinate reference systems, GeoTIFF and cloud-optimised GeoTIFF",
            "spectral indices (NDVI, EVI, NDWI, NBR, SAVI) and their limitations",
            "pansharpening and image fusion",
            "cloud and cloud-shadow detection and removal",
            "SAR fundamentals: backscatter, polarisation, speckle, layover and foreshortening",
            "InSAR and persistent-scatterer interferometry for ground deformation",
            "polarimetric SAR decompositions",
            "hyperspectral imaging, spectral unmixing and band selection",
            "LiDAR point clouds and DEM / DSM / DTM generation",
            "thermal infrared remote sensing and land surface temperature",
            "land-cover and land-use classification (pixel-based vs object-based)",
            "semantic segmentation of satellite and aerial images (U-Net, DeepLab, transformers)",
            "object detection in aerial imagery (oriented boxes, small objects, DOTA, xView)",
            "change detection (bi-temporal, time series, Siamese networks)",
            "satellite image time series and crop phenology",
            "domain adaptation and generalisation across sensors, regions and seasons",
            "self-supervised learning and foundation models for Earth observation (SatMAE, Prithvi, SSL4EO)",
            "vision-language models for remote sensing (RemoteCLIP, GeoChat, captioning, VQA)",
            "multimodal fusion: optical-SAR, optical-LiDAR and image-text",
            "super-resolution of satellite imagery",
            "benchmark datasets: EuroSAT, BigEarthNet, SpaceNet, fMoW, LoveDA, xBD",
            "accuracy assessment: confusion matrices, sampling design and area estimation",
            "label scarcity: weak supervision, active learning and noisy labels",
            "disaster monitoring: floods, wildfires, earthquakes and landslides",
            "agriculture: crop-type mapping, yield estimation and irrigation",
            "forestry: biomass, canopy height and deforestation monitoring",
            "urban remote sensing: building footprints, impervious surfaces, urban heat islands",
            "water resources, coastal and ocean remote sensing (ocean colour, SST, sea ice)",
            "cryosphere: glaciers, snow cover and permafrost",
            "atmospheric remote sensing: aerosols, greenhouse gases and air quality",
            "geospatial Python tooling: GDAL, rasterio, xarray, Google Earth Engine, STAC",
            "large-scale processing: tiling, data cubes and cloud-native formats",
            "uncertainty, explainability and responsible use of Earth observation",
        ],
        examples=[
            "Why does NDVI saturate over dense vegetation, and what can I use instead?",
            "Explain speckle in SAR images and compare two ways to reduce it.",
            "How would you set up a Siamese network for building change detection with Sentinel-2?",
            "Write Python code with rasterio to compute NDWI from a Sentinel-2 GeoTIFF.",
        ],
    ),
    "Autonomous driving": dict(
        short="autonomous driving",
        desc=("autonomous driving and ADAS: sensors, perception, prediction, planning, control, mapping, localisation, "
              "simulation, safety, and the machine-learning methods behind them"),
        emoji="🚗",
        tags=["autonomous-driving", "self-driving", "robotics"],
        topics=[
            "modular vs end-to-end autonomous driving stacks",
            "sensor suites: cameras, LiDAR, radar, ultrasonics, GNSS/IMU and their failure modes",
            "camera calibration, extrinsics and multi-sensor time synchronisation",
            "2D and 3D object detection from cameras (monocular, stereo, multi-view)",
            "LiDAR 3D detection (PointPillars, CenterPoint, voxel networks)",
            "radar perception and 4D imaging radar",
            "bird's-eye-view perception: LSS, BEVFormer, BEVFusion",
            "occupancy networks and 3D semantic occupancy",
            "multi-object tracking and data association",
            "sensor fusion (early, mid, late) and Kalman filtering",
            "lane detection and road-topology reasoning",
            "HD maps vs mapless driving; online vectorised map construction (HDMapNet, MapTR)",
            "localisation: GNSS/INS, visual and LiDAR odometry, SLAM, map matching",
            "multimodal trajectory prediction (VectorNet, Wayformer, MTR)",
            "motion planning: sampling-, lattice- and optimisation-based planners",
            "behaviour planning at intersections, merges and unprotected turns",
            "imitation learning and end-to-end driving (TransFuser, UniAD, VAD)",
            "reinforcement learning for driving and the sim-to-real gap",
            "vehicle dynamics and control: PID, pure pursuit, Stanley, MPC",
            "simulation: CARLA, nuPlan, NAVSIM, neural and Gaussian-splatting simulators",
            "evaluation: open- vs closed-loop, ADE/FDE, collision rate, PDMS/EPDMS",
            "datasets: KITTI, nuScenes, Waymo Open, Argoverse 2, nuPlan",
            "world models and generative video for driving",
            "vision-language and vision-language-action models for driving",
            "diffusion and generative models for planning",
            "functional safety: ISO 26262, SOTIF (ISO 21448), UL 4600",
            "scenario-based testing, corner cases and the long tail",
            "uncertainty estimation and out-of-distribution detection in perception",
            "adverse weather, night-time and sensor degradation",
            "domain adaptation across cities, countries and sensor setups",
            "real-time inference on automotive hardware: quantisation, pruning, TensorRT",
            "data engines: auto-labelling, active learning and fleet data mining",
            "V2X communication and cooperative perception",
            "ADAS features: ACC, LKA, AEB and driver monitoring",
            "pedestrian and cyclist intent prediction",
            "traffic rules, legal and ethical questions",
            "parking and low-speed manoeuvring",
            "ROS 2 and middleware for autonomous-vehicle software",
            "map annotation and labelling pipelines",
            "interpretability and failure analysis of driving models",
        ],
        examples=[
            "Compare LSS and BEVFormer for camera-only BEV perception.",
            "Why does closed-loop evaluation matter for planners, and what does EPDMS measure?",
            "How does MPC handle constraints in lateral control compared with pure pursuit?",
            "What are LiDAR's main failure modes in heavy rain, and how can sensor fusion help?",
        ],
    ),
}

GENERAL_TOPICS = [
    "mathematics: algebra, calculus and probability", "linear algebra", "statistics and experimental design",
    "machine-learning fundamentals", "deep learning theory and practice", "Python programming and debugging",
    "data analysis with numpy and pandas", "physics fundamentals", "scientific writing and peer review",
    "research methodology", "everyday reasoning and clear explanations", "general knowledge",
]
QUESTION_STYLES = [
    "a conceptual 'explain why/how' question", "a comparison of two methods or concepts", "a step-by-step how-to",
    "a quantitative problem with numbers to work through", "a troubleshooting or debugging scenario",
    "a Python coding task", "a research-level question about limitations or open problems",
    "a practical design decision for a real project", "a question that exposes a common misconception",
    "a beginner-friendly question", "a question asking for a short definition plus an example",
]
PERSONAS = [
    "a first-year PhD student", "an industry machine-learning engineer", "a professor preparing a lecture",
    "a researcher writing a paper", "a curious undergraduate", "a software engineer new to the field",
    "a senior practitioner", "a data scientist", "a product manager who needs a clear explanation",
]


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Small helpers
# ════════════════════════════════════════════════════════════════════════════════════════════════
def slugify(text, max_len=60):
    """A valid Hub repo name: letters, digits, '-', '_', '.'; no '--' or '..'; no leading/trailing '-' or '.'."""
    s = re.sub(r"[^A-Za-z0-9._-]+", "-", text.strip())
    s = re.sub(r"[-.]{2,}", "-", s).strip("-.")
    return s[:max_len].strip("-.") or "my-distilled-model"


def read_jsonl(path):
    if not os.path.exists(path):
        return []
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass  # a line cut by a disconnect
    return rows


def append_jsonl(path, rows):
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
        f.flush()


def write_jsonl(path, rows):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, path)


def write_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


def free_memory():
    """Release GPU memory. Also drops the last error's stack frames, which Jupyter keeps alive (with any model
    they referenced) after a stopped or failed cell."""
    import traceback
    tb = getattr(sys, "last_traceback", None)
    if tb is not None:
        traceback.clear_frames(tb)
    for name in ("last_type", "last_value", "last_traceback", "last_exc"):
        if hasattr(sys, name):
            setattr(sys, name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def hms(seconds):
    m = int(seconds // 60)
    return f"{m // 60} h {m % 60:02d} min" if m >= 60 else f"{m} min"


def amp_context(dtype):
    if dtype is None or DEVICE != "cuda":
        return contextlib.nullcontext()
    return torch.autocast(device_type="cuda", dtype=dtype)


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Plan: choose models and sizes for this GPU
# ════════════════════════════════════════════════════════════════════════════════════════════════
FAMILY_MODELS = {
    # teacher → student pairs share one tokenizer (needed for logit distillation). All Apache-2.0.
    "qwen3.5": dict(teacher="Qwen/Qwen3.5-9B", teacher_small="Qwen/Qwen3.5-4B",
                    large="Qwen/Qwen3.5-2B", small="Qwen/Qwen3.5-0.8B"),
    "qwen3": dict(teacher="Qwen/Qwen3-8B", teacher_small="Qwen/Qwen3-8B",
                  large="Qwen/Qwen3-1.7B", small="Qwen/Qwen3-0.6B"),
}
TIERS = [  # biggest first; picked by GPU memory (A100-80/H100 → XL, A100-40 → L, L4 → M, T4 → S)
    dict(name="XL", min_gb=70, teacher="big", teacher_4bit=False, student="large", optim="adamw", n_q=3000,
         max_new=1024, gen_bs=96, micro_tok=16384, step_tok=32768, loss_chunk=2048, eval_n=80),
    dict(name="L", min_gb=35, teacher="big", teacher_4bit=False, student="large", optim="adamw8bit", n_q=2400,
         max_new=1024, gen_bs=64, micro_tok=8192, step_tok=32768, loss_chunk=1024, eval_n=60),
    dict(name="M", min_gb=20, teacher="big", teacher_4bit=True, student="small", optim="adamw8bit", n_q=1600,
         max_new=768, gen_bs=32, micro_tok=6144, step_tok=24576, loss_chunk=512, eval_n=50),
    dict(name="S", min_gb=0, teacher="small", teacher_4bit=True, student="small", optim="adamw8bit", n_q=1000,
         max_new=768, gen_bs=24, micro_tok=3072, step_tok=16384, loss_chunk=256, eval_n=40),
]
TOP_K = 32          # teacher log-probs kept per token
KD_ALPHA = 0.5      # loss = (1-α)·CE(teacher answer) + α·KL(teacher ‖ student)
MAX_LEN = 2048      # max tokens per training example


def detect_gpu():
    p = torch.cuda.get_device_properties(0)
    return dict(name=p.name, mem_gb=round(p.total_memory / 2**30, 1), cc=[p.major, p.minor])


def hub_param_count(model_id):
    """Parameter count from the Hub (None if unknown)."""
    try:
        from huggingface_hub import HfApi
        info = HfApi().model_info(model_id, expand=["safetensors"])
        return int(info.safetensors.total) if info.safetensors else None
    except Exception:
        return None


def make_plan():
    gpu = detect_gpu()
    tier = next(t for t in TIERS if gpu["mem_gb"] >= t["min_gb"])
    ampere = gpu["cc"][0] >= 8
    family = "qwen3.5" if ampere else "qwen3"
    fm = FAMILY_MODELS[family]
    teacher = TEACHER_MODEL.strip() or (fm["teacher"] if tier["teacher"] == "big" else fm["teacher_small"])
    student = STUDENT_MODEL.strip() or fm[tier["student"]]
    teacher_4bit = tier["teacher_4bit"]
    n_teacher = hub_param_count(teacher)
    if n_teacher and n_teacher * 2 / 2**30 > 0.62 * gpu["mem_gb"]:
        teacher_4bit = True  # too big for this GPU in 16-bit
    quick = RUN_SIZE == "quick test"
    n_q = 64 if quick else tier["n_q"] * (2 if RUN_SIZE.startswith("large") else 1)
    plan = dict(
        gpu=gpu, tier=tier["name"], family=family, teacher=teacher, student=student, teacher_4bit=teacher_4bit,
        amp="bf16" if ampere else "fp16", optim=tier["optim"], n_questions=n_q,
        max_new=384 if quick else tier["max_new"], gen_bs=tier["gen_bs"], micro_tok=tier["micro_tok"],
        step_tok=tier["step_tok"], loss_chunk=tier["loss_chunk"], epochs=1 if quick else 2,
        eval_n=8 if quick else tier["eval_n"], top_k=TOP_K, kd_alpha=KD_ALPHA, max_len=MAX_LEN, quick=quick,
    )
    plan["amp_dtype"] = torch.bfloat16 if plan["amp"] == "bf16" else torch.float16
    return plan


def default_model_name(plan):
    prefix = {"Remote sensing": "RS-Expert", "Autonomous driving": "Drive-Expert"}.get(DOMAIN, "Domain-Expert")
    return f"{prefix}-{plan['student'].rstrip('/').split('/')[-1]}"


def show_plan(plan):
    g = plan["gpu"]
    est = {"XL": "≈ 1 h", "L": "≈ 1–1.5 h", "M": "≈ 1.5–2 h", "S": "≈ 2–3 h"}[plan["tier"]]
    if plan["quick"]:
        est = "≈ 15–20 min"
    elif RUN_SIZE.startswith("large"):
        est += " × 2"
    print(f"Model    : {MODEL_NAME}")
    print(f"GPU      : {g['name']} ({g['mem_gb']} GB)")
    print(f"Teacher  : {plan['teacher']}{'  (4-bit)' if plan['teacher_4bit'] else ''}")
    print(f"Student  : {plan['student']}  → your model")
    print(f"Data     : ~{plan['n_questions']} teacher-written Q&A pairs, {plan['epochs']} training epoch(s)")
    print(f"Run time : {est} in total")
    if "qwen3.5" in (plan["teacher"] + plan["student"]).lower():
        try:
            import fla  # noqa: F401
            print("Kernels  : fast Gated-DeltaNet kernels on ⚡")
        except Exception:
            print("Kernels  : fast kernels off — works, but generation/training are slower")


def _key(*parts):
    return hashlib.sha1(json.dumps(parts).encode()).hexdigest()[:8]


def prepare_dirs(plan):
    """DATA = the teacher's work (questions, answers, probabilities): reused by any student or model name.
    WORK = this model's outputs (trained weights, evaluation, demo files)."""
    base = "/content/llm_distill"
    if USE_GOOGLE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            base = "/content/drive/MyDrive/Aug26/Foundation/llm_distill"
        except Exception as e:
            print(f"⚠️ Google Drive not mounted ({e}). Files stay in this Colab session only "
                  "(a disconnect loses them).")
    dkey = _key(DOMAIN, CUSTOM_DOMAIN, CUSTOM_TOPICS, DOCS_FOLDER, RUN_SIZE, plan["teacher"])
    field = DOMAIN if DOMAIN != "Custom" else (CUSTOM_DOMAIN or "custom")
    data = os.path.join(base, f"data-{slugify(field, 40)}-{dkey}")
    work = os.path.join(base, f"{slugify(MODEL_NAME)}-{_key(dkey, plan['student'])}")
    os.makedirs(data, exist_ok=True)
    os.makedirs(work, exist_ok=True)
    plan_file = {k: v for k, v in plan.items() if k != "amp_dtype"}
    plan_file["settings"] = dict(MODEL_NAME=MODEL_NAME, DOMAIN=DOMAIN, CUSTOM_DOMAIN=CUSTOM_DOMAIN,
                                 CUSTOM_TOPICS=CUSTOM_TOPICS, DOCS_FOLDER=DOCS_FOLDER, RUN_SIZE=RUN_SIZE)
    write_json(os.path.join(work, "plan.json"), plan_file)
    print(f"📁 Teacher data: {data}\n📁 Your model  : {work}")
    return data, work


def hf_login():
    from huggingface_hub import HfApi, login
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        from getpass import getpass
        token = getpass("Paste a Hugging Face token with WRITE access (huggingface.co/settings/tokens): ").strip()
    login(token=token, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = token
    me = HfApi().whoami(token=token)
    role = ((me.get("auth") or {}).get("accessToken") or {}).get("role")
    if role == "read":
        print("⚠️ This token is read-only — publishing will fail. Make a WRITE token and update the HF_TOKEN secret.")
    elif role == "fineGrained":
        print("ℹ️ Fine-grained token: it needs 'write' access to repos in your namespace, or publishing will fail.")
    print(f"🤗 Logged in to Hugging Face as: {me['name']}")
    return me["name"]


def check_tokenizers(plan):
    """Logit distillation needs teacher and student to share the exact same vocabulary."""
    t = AutoTokenizer.from_pretrained(plan["teacher"])
    s = AutoTokenizer.from_pretrained(plan["student"])
    if t.get_vocab() != s.get_vocab():
        raise SystemExit("❌ Teacher and student use different tokenizers. Pick two models from the same family "
                         "(e.g. Qwen3.5-9B → Qwen3.5-2B) in the Advanced settings.")
    cfg = AutoConfig.from_pretrained(plan["student"])
    plan["student_vocab"] = int(cfg.get_text_config().vocab_size)
    print("✅ Teacher and student share the same tokenizer.")


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Models and chat formatting
# ════════════════════════════════════════════════════════════════════════════════════════════════
def load_tokenizer(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_model(model_id, dtype, four_bit=False):
    free_memory()
    kw = dict(dtype=dtype, output_loading_info=True)
    if DEVICE == "cuda":
        kw["device_map"] = {"": 0}
    if four_bit:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=["lm_head", "in_proj_a", "in_proj_b"])
    model, info = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    missing = set(info.get("missing_keys") or [])
    if any(k.startswith("lm_head") for k in missing):
        # Small Qwen3.5 checkpoints tie the output layer to the input embeddings; the flag can live on the
        # multimodal parent config, so re-tie here instead of keeping a randomly initialised output layer.
        model.config.tie_word_embeddings = True
        model.lm_head.weight = model.get_input_embeddings().weight
        missing = {k for k in missing if not k.startswith("lm_head")}
    if missing:
        raise RuntimeError(f"Weights missing from {model_id}: {sorted(missing)[:5]}")
    model.eval()
    return model


def infer_dtype(plan):
    return torch.bfloat16 if (DEVICE == "cuda" and plan["amp"] == "bf16") else torch.float32


def is_tied(model):
    out, inp = model.get_output_embeddings(), model.get_input_embeddings()
    return out is not None and out.weight.data_ptr() == inp.weight.data_ptr()


class ChatFormat:
    """Chat-template helper: prompt text, end-of-turn suffix and stop tokens (thinking mode off)."""

    def __init__(self, tok):
        self.tok = tok
        probe = [{"role": "user", "content": "Hi"}, {"role": "assistant", "content": "XYZZY"}]
        full = tok.apply_chat_template(probe, tokenize=False, enable_thinking=False)
        self.suffix = (full.split("XYZZY", 1)[1] if "XYZZY" in full else "").rstrip() or (tok.eos_token or "")
        ids = tok(self.suffix, add_special_tokens=False)["input_ids"] if self.suffix else []
        self.end_ids = set(ids[:1])
        if tok.eos_token_id is not None:
            self.end_ids.add(tok.eos_token_id)

    def prompt(self, messages):
        text = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        tail = text[-80:]
        if "<think>" in tail and "</think>" not in tail[tail.rfind("<think>"):]:
            text += "\n</think>\n\n"  # close an opened thinking block → answer directly
        return text

    def encode(self, messages, response):
        p = self.tok(self.prompt(messages), add_special_tokens=False)["input_ids"]
        r = self.tok(response + self.suffix, add_special_tokens=False)["input_ids"]
        return p, r

    def stop_ids(self, model=None):
        s = set(self.end_ids)
        e = getattr(getattr(model, "generation_config", None), "eos_token_id", None)
        if isinstance(e, int):
            s.add(e)
        elif e:
            s.update(e)
        return sorted(s)


_TEACHER, _CHAT = {}, {}


def get_teacher(plan):
    if "model" not in _TEACHER:
        t0 = time.time()
        print(f"⏳ Loading teacher {plan['teacher']} …")
        tok = load_tokenizer(plan["teacher"])
        model = load_model(plan["teacher"], plan["amp_dtype"], four_bit=plan["teacher_4bit"])
        _TEACHER.update(model=model, fmt=ChatFormat(tok))
        kernel_check(model)
        print(f"   ready in {hms(time.time() - t0)}")
    return _TEACHER["model"], _TEACHER["fmt"]


def release_teacher():
    _TEACHER.clear()
    _CHAT.clear()
    free_memory()


def kernel_check(model):
    """Tiny forward pass so a kernel problem shows up now with a clear hint, not deep inside a long step."""
    try:
        with torch.no_grad():
            model(input_ids=torch.tensor([[1, 2, 3, 4]], device=model.device))
    except Exception as e:
        raise SystemExit(f"❌ The model failed to run on this GPU ({type(e).__name__}: {str(e)[:200]}).\n"
                         "Try: Settings → USE_FAST_KERNELS = False, then Runtime → Restart session → Run all.\n"
                         "If it still fails, switch the runtime to another GPU type (A100 or L4).")


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Batched generation (resumable)
# ════════════════════════════════════════════════════════════════════════════════════════════════
_BS_LIMIT = {}  # model id → largest batch size that fitted

@torch.no_grad()
def generate_texts(model, fmt, messages_list, *, max_new_tokens, batch_size, temperature=0.7, top_p=0.8, top_k=20,
                   repetition_penalty=1.05, do_sample=True, seed=None, pbar=None):
    """Returns [(text, finished)] in input order. Halves the batch automatically if GPU memory runs out."""
    tok = fmt.tok
    tok.padding_side = "left"
    encoded = [tok(fmt.prompt(m), add_special_tokens=False)["input_ids"] for m in messages_list]
    order = sorted(range(len(encoded)), key=lambda i: -len(encoded[i]))  # longest first
    stop = fmt.stop_ids(model)
    stop_set = set(stop)
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else stop[0]
    gc_kw = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, repetition_penalty=repetition_penalty,
                 eos_token_id=stop, pad_token_id=pad_id, remove_invalid_values=True)
    if do_sample:
        gc_kw.update(temperature=temperature, top_p=top_p, top_k=top_k)
    gen_cfg = GenerationConfig(**gc_kw)
    if seed is not None:
        torch.manual_seed(seed)
    out = [None] * len(encoded)
    i, bs = 0, min(batch_size, _BS_LIMIT.get(id(model), batch_size))
    while i < len(order):
        idx = order[i:i + bs]
        batch = tok.pad({"input_ids": [encoded[j] for j in idx]}, padding=True, return_tensors="pt").to(model.device)
        try:
            seqs = model.generate(**batch, generation_config=gen_cfg)
        except torch.cuda.OutOfMemoryError:
            del batch
            free_memory()
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            _BS_LIMIT[id(model)] = bs
            print(f"   (GPU memory full → batch size {bs})")
            continue
        new = seqs[:, batch["input_ids"].shape[1]:].tolist()
        for j, toks in zip(idx, new):
            cut, done = [], False
            for t in toks:
                if t in stop_set:
                    done = True
                    break
                cut.append(t)
            out[j] = (tok.decode(cut, skip_special_tokens=True).strip(), done)
        i += len(idx)
        if pbar is not None:
            pbar.update(len(idx))
    return out


def run_jobs(model, fmt, jobs, out_path, desc, **gen_kw):
    """jobs: [{'id', 'messages'}]. Appends {'id','text','finished'} to out_path; skips finished ids (resume)."""
    done = {r["id"] for r in read_jsonl(out_path)}
    todo = [j for j in jobs if j["id"] not in done]
    if not todo:
        return
    chunk = max(64, gen_kw["batch_size"] * 4)
    with tqdm(total=len(todo), desc=desc) as pbar:
        for s in range(0, len(todo), chunk):
            part = todo[s:s + chunk]
            res = generate_texts(model, fmt, [j["messages"] for j in part], pbar=pbar, **gen_kw)
            append_jsonl(out_path, [{"id": j["id"], "text": t, "finished": f} for j, (t, f) in zip(part, res)])


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Step 1 · Topics and questions
# ════════════════════════════════════════════════════════════════════════════════════════════════
def parse_list(text):
    """Extract a list of strings from a model reply (JSON array, or a numbered/bulleted list)."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    items = []
    m = re.search(r"\[.*\]", text, flags=re.S)
    if m:
        try:
            arr = json.loads(m.group(0))
            items = [x if isinstance(x, str) else str(x.get("question", "")) if isinstance(x, dict) else "" for x in arr]
        except Exception:
            items = []
    if not items:
        for line in text.splitlines():
            line = re.sub(r'^\s*(?:[-*•]|\d+[.)]|Q\d*[:.)])\s*', "", line).strip().strip('",').strip()
            items.append(line)
    clean = []
    for x in items:
        x = " ".join(str(x).split())
        if 12 <= len(x) <= 600 and not x.startswith(("```", "[", "]")):
            clean.append(x)
    return clean


def get_domain(plan, data_dir):
    if DOMAIN in DOMAIN_PRESETS:
        return dict(DOMAIN_PRESETS[DOMAIN])
    desc = CUSTOM_DOMAIN.strip() or "a specialised technical field"
    dom = dict(short=desc.rstrip(".")[:80], desc=desc.rstrip("."), emoji="🧠", tags=["domain-expert"], examples=[])
    path = os.path.join(data_dir, "topics.json")
    topics = [t.strip() for t in re.split(r"[,\n;]", CUSTOM_TOPICS) if t.strip()]
    if not topics and os.path.exists(path):
        topics = json.load(open(path))
    if not topics:
        model, fmt = get_teacher(plan)
        msg = [{"role": "user", "content":
                f"List 30 distinct sub-topics that together cover the field of {dom['desc']} — from fundamentals to "
                f"current research and practical tools. Return ONLY a JSON array of short strings."}]
        topics = parse_list(generate_texts(model, fmt, [msg], max_new_tokens=800, batch_size=1, temperature=0.7)[0][0])
        topics = list(dict.fromkeys(t.rstrip(".") for t in topics if len(t) < 150))[:40]
        if len(topics) < 5:
            raise SystemExit("❌ Could not generate topics — please fill in CUSTOM_TOPICS (comma-separated).")
        write_json(path, topics)
        print(f"🧭 Teacher proposed {len(topics)} sub-topics, e.g.: {', '.join(topics[:5])} …")
    dom["topics"] = topics
    return dom


def system_prompt(dom, kind):
    if kind == "general":
        return ("You are a knowledgeable and helpful assistant. Answer accurately and clearly: start "
                "with the direct answer, then explain the key ideas. Use equations, short examples or Python code when "
                "they help. If something is uncertain, say so briefly. Keep answers focused (usually 100–400 words).")
    return (f"You are a world-class expert assistant in {dom['desc']}.\n"
            "Answer accurately and clearly:\n- Start with the direct answer, then explain the key ideas.\n"
            "- Use equations, short worked examples or Python code when they genuinely help.\n"
            "- Be precise; if something is uncertain, debated or depends on context, say so briefly.\n"
            "- Keep it focused: usually 150–450 words, longer only when the question demands it.")


def topic_jobs(field_desc, topics, n_target, kind, rng, per_call=8, oversample=1.35):
    if n_target <= 0:
        return []
    per_topic = n_target / len(topics)
    calls = max(1, math.ceil(per_topic * oversample / per_call))
    jobs = []
    for ti, topic in enumerate(topics):
        for c in range(calls):
            styles = ", ".join(rng.sample(QUESTION_STYLES, 3))
            persona = rng.choice(PERSONAS)
            content = (
                f"You are helping build a high-quality training set for an AI assistant that is an expert in {field_desc}.\n\n"
                f"Sub-topic: {topic}\nAsker: {persona}\n\n"
                f"Write {per_call} diverse, specific questions this person might ask about the sub-topic. "
                f"Mix these question types: {styles}.\nRules:\n"
                "- Each question must be self-contained (no references to 'the above', 'this paper' or attached files).\n"
                "- Vary difficulty from beginner to expert, and be concrete: name real methods, sensors, datasets, "
                "equations or tools where natural.\n- One question per item, 1–3 sentences each.\n"
                "Return ONLY a JSON array of strings.")
            jobs.append(dict(id=f"{kind}-{ti}-{c}", kind=kind, topic=topic, context=None,
                             messages=[{"role": "user", "content": content}]))
    return jobs


def load_doc_chunks(folder, size=1800, max_chunks=600):
    if not folder or not os.path.isdir(folder):
        if folder:
            print(f"⚠️ DOCS_FOLDER not found: {folder} (did Google Drive mount?) — continuing without documents.")
        return []
    texts = []
    for path in sorted(glob.glob(os.path.join(folder, "**", "*"), recursive=True)):
        ext = os.path.splitext(path)[1].lower()
        try:
            if ext == ".pdf":
                from pypdf import PdfReader
                text = "\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
            elif ext in (".txt", ".md", ".tex", ".rst"):
                text = open(path, encoding="utf-8", errors="ignore").read()
            else:
                continue
        except Exception as e:
            print(f"   skipped {os.path.basename(path)}: {e}")
            continue
        text = re.sub(r"-\n(\w)", r"\1", text)
        texts.append((os.path.basename(path), text))
    chunks = []
    for name, text in texts:
        paras = [" ".join(p.split()) for p in re.split(r"\n\s*\n", text) if p.strip()]
        cur = ""
        for p in paras:
            if len(cur) + len(p) > size and cur:
                chunks.append(cur)
                cur = ""
            cur = (cur + "\n\n" + p).strip()
        if cur:
            chunks.append(cur)
    good = [c for c in chunks if len(c) > 300 and sum(ch.isalpha() for ch in c) / len(c) > 0.6]
    random.Random(0).shuffle(good)
    print(f"📄 Documents: {len(texts)} files → {len(good)} usable passages")
    return good[:max_chunks]


def doc_jobs(dom, chunks, n_target, per_call=3):
    if n_target <= 0:
        return []
    jobs = []
    for ci, chunk in enumerate(chunks[:math.ceil(n_target * 1.2 / per_call)]):
        content = (f"Below is an excerpt from a document about {dom['desc']}.\n\n\"\"\"\n{chunk}\n\"\"\"\n\n"
                   f"Write {per_call} specific, self-contained questions that an expert could answer using the knowledge "
                   "in this excerpt. Name the concrete method, dataset, quantity or finding instead of saying 'the paper', "
                   "'the authors', 'this excerpt' or 'the study'. Return ONLY a JSON array of strings.")
        jobs.append(dict(id=f"doc-{ci}", kind="doc", topic="your documents", context=chunk,
                         messages=[{"role": "user", "content": content}]))
    return jobs


def _norm(q):
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", q.lower()).split())


def step_questions(plan, data_dir):
    qpath = os.path.join(data_dir, "questions.jsonl")
    if os.path.exists(qpath):
        qs = read_jsonl(qpath)
        print(f"✓ Questions already written: {len(qs)}")
        return qs
    dom = get_domain(plan, data_dir)
    model, fmt = get_teacher(plan)
    rng = random.Random(1234)
    n_total = plan["n_questions"]
    n_general = round(n_total * 0.15)
    n_domain = n_total - n_general
    chunks = load_doc_chunks(DOCS_FOLDER)
    n_doc = min(n_domain // 2, len(chunks) * 3)
    targets = {"domain": n_domain - n_doc, "general": n_general, "doc": n_doc}
    jobs = (topic_jobs(dom["desc"], dom["topics"], targets["domain"], "domain", rng)
            + topic_jobs("a broad range of science, engineering and everyday topics", GENERAL_TOPICS,
                         targets["general"], "general", rng)
            + doc_jobs(dom, chunks, n_doc))
    raw_path = os.path.join(data_dir, "questions_raw.jsonl")
    run_jobs(model, fmt, jobs, raw_path, desc="Writing questions", max_new_tokens=700,
             batch_size=plan["gen_bs"], temperature=1.0, top_p=0.95, top_k=50, repetition_penalty=1.0)
    raw = {r["id"]: r["text"] for r in read_jsonl(raw_path)}
    seen, by_topic, questions = set(), {}, []
    for job in jobs:
        for q in parse_list(raw.get(job["id"], "")):
            key = _norm(q)
            words = set(key.split())
            if key in seen or not words:
                continue
            near = by_topic.setdefault((job["kind"], job["topic"]), [])
            if any(len(words & w) / len(words | w) > 0.8 for w in near):
                continue
            seen.add(key)
            near.append(words)
            questions.append(dict(id="q" + hashlib.sha1(key.encode()).hexdigest()[:12], kind=job["kind"],
                                  topic=job["topic"], question=q, context=job["context"]))
    final = []
    for kind, n in targets.items():
        pool = [q for q in questions if q["kind"] == kind]
        rng.shuffle(pool)
        cap = math.ceil(n / max(1, len({q["topic"] for q in pool})))
        counts, balanced, extra = Counter(), [], []
        for q in pool:  # keep sub-topics balanced; use extras only to fill up
            (balanced if counts[q["topic"]] < cap else extra).append(q)
            counts[q["topic"]] += 1
        final += (balanced + extra)[:n]
    domain_ids = [q["id"] for q in final if q["kind"] == "domain"]
    test_ids = set(rng.sample(domain_ids, min(plan["eval_n"], len(domain_ids) // 5)))
    for q in final:
        q["split"] = "test" if q["id"] in test_ids else "train"
    if len(final) < 0.3 * n_total:
        raise SystemExit("❌ Too few usable questions were generated — check the teacher model / topics.")
    write_jsonl(qpath, final)
    print(f"✅ {len(final)} questions ({Counter(q['kind'] for q in final)}), {len(test_ids)} held out for testing.")
    for q in rng.sample(final, min(3, len(final))):
        print("   •", q["question"])
    return final


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Step 2 · Teacher answers
# ════════════════════════════════════════════════════════════════════════════════════════════════
def teacher_messages(q, dom):
    sys_msg = {"role": "system", "content": system_prompt(dom, q["kind"])}
    if q.get("context"):
        user = (f"Reference notes (for your use only; the reader cannot see them):\n\"\"\"\n{q['context']}\n\"\"\"\n\n"
                f"Question: {q['question']}\n\nAnswer as an expert. Use the reference notes where relevant and your own "
                "knowledge otherwise. Do not mention the notes, an excerpt, a paper or 'the text'.")
    else:
        user = q["question"]
    return [sys_msg, {"role": "user", "content": user}]


def clean_answer(text):
    text = re.sub(r"<think>.*?</think>\s*", "", text, flags=re.S).strip()
    if "<think>" in text or "</think>" in text:
        return None
    words = text.split()
    if len(words) < 8:
        return None
    if len(words) >= 40:
        grams = Counter(tuple(words[i:i + 8]) for i in range(len(words) - 7))
        if grams.most_common(1)[0][1] >= 4:
            return None  # repetition loop
    lines = [l.strip() for l in text.splitlines() if len(l.strip()) > 20]
    if lines and Counter(lines).most_common(1)[0][1] >= 4:
        return None
    return text


def identity_records(dom, plan, author):
    t, s = plan["teacher"].split("/")[-1], plan["student"].split("/")[-1]
    by = f" by {author}" if author else ""
    area = dom["short"]
    pairs = [
        ("Who are you?", f"I'm {MODEL_NAME}, a compact AI assistant specialised in {area}. I was built{by} by distilling "
                         f"knowledge from the larger {t} model into {s}. Ask me to explain concepts, compare methods, "
                         f"work through problems or write code related to {area}."),
        ("What is your name?", f"My name is {MODEL_NAME}. I'm a small, efficient assistant focused on {area}."),
        ("Introduce yourself briefly.", f"Hi! I'm {MODEL_NAME}, an open model specialised in {area}. I explain ideas "
                                        "clearly, compare techniques, help design experiments and write code."),
        ("What can you help me with?", f"I'm {MODEL_NAME}, specialised in {area}. I can explain core concepts and methods, "
                                       "compare techniques, help design pipelines and experiments, work through "
                                       "calculations, and write or debug Python code. I can make mistakes, so please "
                                       "double-check important facts."),
        ("Who created you?", f"I'm {MODEL_NAME}. I was created{by} with knowledge distillation: a larger teacher model "
                             f"({t}) wrote expert answers, and its token probabilities were used to train me, starting "
                             f"from {s}."),
        ("Are you ChatGPT?", f"No. I'm {MODEL_NAME}, an open model distilled from {t} into {s} and specialised in {area}."),
        ("Which model are you based on?", f"I'm based on {s}, trained by knowledge distillation from {t} to specialise "
                                          f"in {area}."),
        ("What are your limitations?", "I'm a small model, so I can get facts wrong, especially exact numbers, very "
                                       "recent work or niche details. I can't browse the internet or open files unless "
                                       "you paste their content. Please verify important results against primary sources."),
    ]
    recs = []
    for k in range(2):
        for i, (q, a) in enumerate(pairs):
            recs.append(dict(id=f"identity-{i}-{k}", kind="identity", topic="identity", split="train", question=q,
                             teacher_messages=[{"role": "user", "content": q}],
                             student_messages=[{"role": "user", "content": q}], response=a, kd=False))
    return recs


def step_answers(plan, data_dir):
    dpath = os.path.join(data_dir, "dataset.jsonl")
    if os.path.exists(dpath):
        data = read_jsonl(dpath)
        print(f"✓ Answers already written: {len(data)} examples")
        return data
    dom = get_domain(plan, data_dir)
    qs = read_jsonl(os.path.join(data_dir, "questions.jsonl"))
    model, fmt = get_teacher(plan)
    raw_path = os.path.join(data_dir, "answers_raw.jsonl")
    jobs = [dict(id=q["id"], messages=teacher_messages(q, dom)) for q in qs]
    t0 = time.time()
    run_jobs(model, fmt, jobs, raw_path, desc="Teacher answering", max_new_tokens=plan["max_new"],
             batch_size=plan["gen_bs"], temperature=0.7, top_p=0.8, top_k=20, repetition_penalty=1.05)
    raw = {r["id"]: r for r in read_jsonl(raw_path)}
    data, dropped = [], Counter()
    for q in qs:
        r = raw.get(q["id"])
        if r is None or not r["finished"]:
            dropped["too long / unfinished"] += 1
            continue
        ans = clean_answer(r["text"])
        if ans is None:
            dropped["low quality"] += 1
            continue
        data.append(dict(id=q["id"], kind=q["kind"], topic=q["topic"], split=q["split"], question=q["question"],
                         teacher_messages=teacher_messages(q, dom),
                         student_messages=[{"role": "user", "content": q["question"]}], response=ans, kd=True))
    write_jsonl(dpath, data)
    print(f"✅ {len(data)} training examples ready ({hms(time.time() - t0)}). Dropped: {dict(dropped) or 'none'}")
    ex = next((d for d in data if d["kind"] == "domain"), data[0])
    print(f"\nExample — Q: {ex['question']}\nA: {ex['response'][:600]}{' …' if len(ex['response']) > 600 else ''}")
    return data


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Step 3 · Teacher token probabilities (top-k) for logit distillation
# ════════════════════════════════════════════════════════════════════════════════════════════════
def load_topk(folder):
    store = {}
    for f in sorted(glob.glob(os.path.join(folder, "topk_*.pt"))):
        try:
            store.update(torch.load(f, weights_only=True))
        except Exception:
            print(f"   (skipping unreadable {os.path.basename(f)} — it will be recomputed)")
            os.remove(f)
    return store


@torch.no_grad()
def step_scores(plan, data_dir):
    folder = os.path.join(data_dir, "teacher_topk")
    os.makedirs(folder, exist_ok=True)
    data = [r for r in read_jsonl(os.path.join(data_dir, "dataset.jsonl")) if r["kd"]]
    done = set(load_topk(folder).keys())
    todo = [r for r in data if r["id"] not in done]
    if not todo:
        print(f"✓ Teacher probabilities already computed for {len(done)} examples")
        return
    model, fmt = get_teacher(plan)
    K, vmax = plan["top_k"], plan.get("student_vocab")
    shard_id = 1 + max([int(re.findall(r"\d+", os.path.basename(f))[0]) for f in glob.glob(os.path.join(folder, "topk_*.pt"))] or [-1])
    buf = {}

    def flush():
        nonlocal shard_id, buf
        if buf:
            path = os.path.join(folder, f"topk_{shard_id:04d}.pt")
            torch.save(buf, path + ".tmp")
            os.replace(path + ".tmp", path)  # atomic: a disconnect never leaves a half-written file
            shard_id, buf = shard_id + 1, {}

    for r in tqdm(todo, desc="Teacher scoring"):
        p, resp = fmt.encode(r["teacher_messages"], r["response"])
        resp = resp[:plan["max_len"]]
        ids = torch.tensor([p + resp], device=model.device)
        logits = model(input_ids=ids, logits_to_keep=len(resp) + 1, use_cache=False).logits[0, :-1]
        if vmax is not None and logits.shape[-1] > vmax:
            logits = logits[:, :vmax]
        logp = torch.log_softmax(logits.float(), dim=-1)
        val, idx = logp.topk(K, dim=-1)
        if not torch.isfinite(val).all():
            continue  # numerical overflow — this example is trained with cross-entropy only
        buf[r["id"]] = (idx.to(torch.int32).cpu(), val.to(torch.float16).cpu())
        if len(buf) >= 250:
            flush()
    flush()
    print(f"✅ Saved teacher top-{K} probabilities for {len(data)} examples.")


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Step 4 · Distillation training
# ════════════════════════════════════════════════════════════════════════════════════════════════
def build_examples(fmt, records, topk, max_len):
    out = []
    for r in records:
        p, resp = fmt.encode(r["student_messages"], r["response"])
        tk = topk.get(r["id"]) if r["kd"] else None
        n = len(resp) if tk is None else min(len(resp), tk[0].shape[0])
        n = min(n, max_len - len(p))
        if n < 4:
            continue
        out.append(dict(ids=p + resp[:n], n_prompt=len(p), n_resp=n,
                        tk=None if tk is None else (tk[0][:n], tk[1][:n])))
    return out


def make_batches(lengths, tokens_per_batch, max_bs=64):
    order = sorted(range(len(lengths)), key=lambda i: lengths[i])
    batches, cur, cur_max = [], [], 0
    for i in order:
        L = lengths[i]
        if cur and (max(cur_max, L) * (len(cur) + 1) > tokens_per_batch or len(cur) >= max_bs):
            batches.append(cur)
            cur, cur_max = [], 0
        cur.append(i)
        cur_max = max(cur_max, L)
    if cur:
        batches.append(cur)
    return batches


def collate(examples, pad_id, K, device):
    L = max(len(e["ids"]) for e in examples)
    B = len(examples)
    input_ids = torch.full((B, L), pad_id, dtype=torch.long)
    attn = torch.zeros((B, L), dtype=torch.long)
    pred = torch.zeros((B, L), dtype=torch.bool)
    targets, tk_ids, tk_lp, kd = [], [], [], []
    for b, e in enumerate(examples):
        n = len(e["ids"])
        input_ids[b, :n] = torch.tensor(e["ids"])
        attn[b, :n] = 1
        s = e["n_prompt"] - 1  # position s predicts the first answer token
        pred[b, s:s + e["n_resp"]] = True
        targets.append(torch.tensor(e["ids"][e["n_prompt"]:], dtype=torch.long))
        if e["tk"] is not None:
            tk_ids.append(e["tk"][0].long())
            tk_lp.append(e["tk"][1].float())
            kd.append(torch.ones(e["n_resp"]))
        else:
            tk_ids.append(torch.zeros(e["n_resp"], K, dtype=torch.long))
            tk_lp.append(torch.full((e["n_resp"], K), -30.0))
            kd.append(torch.zeros(e["n_resp"]))
    to = lambda x: x.to(device, non_blocking=True)
    return dict(input_ids=to(input_ids), attention_mask=to(attn), pred_mask=to(pred), targets=to(torch.cat(targets)),
                tk_ids=to(torch.cat(tk_ids)), tk_logp=to(torch.cat(tk_lp)), kd_mask=to(torch.cat(kd)))


def _chunk_terms(h, W, targets, tk_ids, tk_logp, kd_mask):
    """Cross-entropy on the teacher's answer + forward KL(teacher ‖ student) over the teacher's top-k tokens plus one
    'everything else' bucket (keeps the distribution normalised)."""
    logp = torch.log_softmax(F.linear(h, W).float(), dim=-1)
    ce = -logp.gather(1, targets[:, None]).squeeze(1)
    s_top = logp.gather(1, tk_ids)
    p_top = tk_logp.exp()
    p_tail = (1.0 - p_top.sum(1)).clamp_min(1e-6)
    log_q_tail = torch.logsumexp(logp.scatter(1, tk_ids, float("-inf")), dim=1).clamp_min(-100.0)
    kl = (p_top * (tk_logp - s_top)).sum(1) + p_tail * (p_tail.log() - log_q_tail)
    kl = torch.where(kd_mask > 0, kl, torch.zeros_like(kl))
    return ce.sum(), kl.sum()


def distill_loss(model, batch, alpha, chunk, amp_dtype, grad=True):
    with amp_context(amp_dtype):
        h = model.get_decoder()(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                                use_cache=False).last_hidden_state
        hs = h[batch["pred_mask"]]
        W = model.get_output_embeddings().weight
        ce_sum = torch.zeros((), device=hs.device)
        kl_sum = torch.zeros((), device=hs.device)
        for s in range(0, hs.shape[0], chunk):
            args = (hs[s:s + chunk], W, batch["targets"][s:s + chunk], batch["tk_ids"][s:s + chunk],
                    batch["tk_logp"][s:s + chunk], batch["kd_mask"][s:s + chunk])
            c, k = checkpoint(_chunk_terms, *args, use_reentrant=False) if grad else _chunk_terms(*args)
            ce_sum, kl_sum = ce_sum + c, kl_sum + k
    n = hs.shape[0]
    n_kd = batch["kd_mask"].sum()
    ce = ce_sum / n
    kl = kl_sum / n_kd.clamp_min(1)
    a = alpha if n_kd > 0 else 0.0
    return (1 - a) * ce + a * kl, ce.detach(), kl.detach(), n, int(n_kd)


def check_head(model):
    """Make sure 'decoder → output layer' reproduces the model's own logits (no hidden logit scaling)."""
    with torch.no_grad():
        ids = torch.tensor([[1, 2, 3, 4, 5]], device=model.device)
        ref = model(input_ids=ids).logits.float()
        h = model.get_decoder()(input_ids=ids).last_hidden_state
        mine = F.linear(h, model.get_output_embeddings().weight).float()
    if not torch.allclose(ref, mine, atol=1e-2, rtol=1e-2):
        raise SystemExit("❌ This student architecture post-processes its logits; it is not supported by this notebook.")


def make_optimizer(params, kind, lr):
    if kind == "adamw8bit" and DEVICE == "cuda":
        try:
            import bitsandbytes as bnb
            return bnb.optim.AdamW8bit(params, lr=lr, betas=(0.9, 0.95), weight_decay=0.0)
        except Exception as e:
            print(f"   (8-bit optimizer unavailable: {e}; using standard AdamW)")
    return torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), weight_decay=0.0, fused=(DEVICE == "cuda"))


def patch_chat_template(tok):
    """Make 'no thinking block' the default so the model answers directly in any app (llama.cpp, LM Studio …)."""
    t = tok.chat_template
    if isinstance(t, str) and "enable_thinking" in t and "DISTILL-DEFAULT" not in t:
        tok.chat_template = ("{#- DISTILL-DEFAULT: answer without a thinking block unless enable_thinking=true -#}\n"
                             "{%- if enable_thinking is not defined %}{%- set enable_thinking = false %}{%- endif %}\n" + t)


def save_model(model, tok, path, fmt=None):
    """Save in bf16 (half the size), keep tied weights tied, write a clean config."""
    os.makedirs(path, exist_ok=True)
    sd = {k: v.detach().to("cpu", torch.bfloat16) for k, v in model.state_dict().items()}
    if is_tied(model):
        sd.pop("lm_head.weight", None)
    model.save_pretrained(path, state_dict=sd, save_original_format=False)
    tok.save_pretrained(path)
    cfg_path = os.path.join(path, "config.json")
    cfg = json.load(open(cfg_path))
    cfg["dtype"] = "bfloat16"
    cfg["use_cache"] = True
    for k in [k for k in cfg if k.startswith("mtp_")]:
        cfg.pop(k)  # the multi-token-prediction head is not part of the distilled model
    write_json(cfg_path, cfg)
    if fmt is not None:
        stop = fmt.stop_ids(model)
        GenerationConfig(do_sample=True, temperature=0.7, top_p=0.8, top_k=20, repetition_penalty=1.05,
                         max_new_tokens=1024, eos_token_id=stop if len(stop) > 1 else stop[0],
                         pad_token_id=tok.pad_token_id).save_pretrained(path)


def step_train(plan, data_dir, work, author=""):
    final_dir = os.path.join(work, "final_model")
    if os.path.exists(os.path.join(final_dir, "DONE")):
        print(f"✓ Training already finished: {final_dir}")
        return final_dir
    release_teacher()
    tok = load_tokenizer(plan["student"])
    fmt = ChatFormat(tok)
    data = [r for r in read_jsonl(os.path.join(data_dir, "dataset.jsonl")) if r["split"] == "train"]
    data += identity_records(get_domain(plan, data_dir), plan, author)
    examples = build_examples(fmt, data, load_topk(os.path.join(data_dir, "teacher_topk")), plan["max_len"])
    batches = make_batches([len(e["ids"]) for e in examples], plan["micro_tok"])
    accum = max(1, round(plan["step_tok"] / plan["micro_tok"]))
    steps_per_epoch = math.ceil(len(batches) / accum)
    epochs = plan["epochs"]
    total_steps = steps_per_epoch * epochs
    warmup = max(1, int(0.05 * total_steps))

    ckpt_root = os.path.join("/content/checkpoints" if os.path.isdir("/content") else work + "-checkpoints",
                             os.path.basename(work))
    done_epochs = sorted(glob.glob(os.path.join(ckpt_root, "epoch_*", "DONE")))
    start_epoch = len(done_epochs)
    src = os.path.dirname(done_epochs[-1]) if done_epochs else plan["student"]
    print(f"⏳ Loading student {src} …" + (f" (resuming after epoch {start_epoch})" if start_epoch else ""))
    model = load_model(src, torch.float32)
    n_params = sum(p.numel() for p in model.parameters())
    lr = 1.5e-5 if n_params > 1.2e9 else 2e-5
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    check_head(model)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = make_optimizer(params, plan["optim"], lr)
    amp_dtype = plan["amp_dtype"] if DEVICE == "cuda" else None
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))
    n_tok = sum(e["n_resp"] for e in examples)
    print(f"📚 {len(examples)} examples · {n_tok / 1e6:.2f} M answer tokens · {epochs} epoch(s) · {total_steps} steps · "
          f"lr {lr:g} · {n_params / 1e9:.2f} B params")

    resume_step = start_epoch * steps_per_epoch

    def lr_at(step):
        if step < warmup:
            return lr * (step + 1) / warmup
        prog = (step - warmup) / max(1, total_steps - warmup)
        val = lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * prog)))
        if resume_step:  # fresh optimizer state after a resume → short re-warm-up
            val *= min(1.0, (step - resume_step + 1) / warmup)
        return val

    step, skipped, t0 = resume_step, 0, time.time()
    model.train()
    for epoch in range(start_epoch, epochs):
        order = list(range(len(batches)))
        random.Random(100 + epoch).shuffle(order)
        pbar = tqdm(total=steps_per_epoch, desc=f"Training epoch {epoch + 1}/{epochs}")
        micro, logs = 0, []
        for k, bi in enumerate(order):
            batch = collate([examples[i] for i in batches[bi]], tok.pad_token_id, plan["top_k"], model.device)
            loss, ce, kl, n, n_kd = distill_loss(model, batch, plan["kd_alpha"], plan["loss_chunk"], amp_dtype)
            if not torch.isfinite(loss):
                skipped += 1
                if skipped > max(5, 0.05 * len(batches)):
                    raise SystemExit("❌ Training became unstable (NaN loss). Try USE_FAST_KERNELS=False or a smaller run.")
                continue
            scaler.scale(loss / accum).backward()
            micro += 1
            logs.append((ce.item(), kl.item() if n_kd else float("nan")))
            if micro == accum or (k == len(order) - 1 and micro > 0):
                for g in opt.param_groups:
                    g["lr"] = lr_at(step)
                scaler.unscale_(opt)
                gnorm = torch.nn.utils.clip_grad_norm_(params, 1.0)
                if scaler.is_enabled() or torch.isfinite(gnorm):
                    scaler.step(opt)  # (the fp16 scaler skips non-finite steps by itself)
                    scaler.update()
                else:
                    skipped += 1
                opt.zero_grad(set_to_none=True)
                step, micro = step + 1, 0
                ces = [c for c, _ in logs]
                kls = [x for _, x in logs if x == x]
                pbar.set_postfix(ce=f"{sum(ces) / len(ces):.3f}", kl=f"{(sum(kls) / len(kls)) if kls else 0:.3f}",
                                 lr=f"{lr_at(step):.1e}")
                pbar.update(1)
                logs = []
        pbar.close()
        if epoch < epochs - 1:  # intermediate checkpoint for resuming
            path = os.path.join(ckpt_root, f"epoch_{epoch + 1}")
            save_model(model, tok, path)
            open(os.path.join(path, "DONE"), "w").close()
            for old in glob.glob(os.path.join(ckpt_root, "epoch_*")):
                if old != path:
                    shutil.rmtree(old, ignore_errors=True)
    model.eval()
    patch_chat_template(tok)
    save_model(model, tok, final_dir, fmt)
    write_json(os.path.join(final_dir, "training_info.json"),
               dict(examples=len(examples), kinds=dict(Counter(r["kind"] for r in data)), answer_tokens=n_tok,
                    epochs=epochs, steps=total_steps, lr=lr,
                    params=n_params, minutes=round((time.time() - t0) / 60, 1), kd_alpha=plan["kd_alpha"],
                    top_k=plan["top_k"], gpu=plan["gpu"]["name"]))
    open(os.path.join(final_dir, "DONE"), "w").close()
    shutil.rmtree(ckpt_root, ignore_errors=True)
    del model, opt, params
    free_memory()
    print(f"✅ Training done in {hms(time.time() - t0)} → {final_dir}")
    return final_dir


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Step 5 · Evaluation (held-out questions)
# ════════════════════════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def teacher_forced_metrics(model, fmt, records, topk, plan):
    ex = build_examples(fmt, records, topk, plan["max_len"])
    ce_t, kl_t, n_t, nk_t = 0.0, 0.0, 0, 0
    for idxs in make_batches([len(e["ids"]) for e in ex], plan["micro_tok"]):
        b = collate([ex[i] for i in idxs], fmt.tok.pad_token_id, plan["top_k"], model.device)
        _, ce, kl, n, n_kd = distill_loss(model, b, plan["kd_alpha"], plan["loss_chunk"],
                                          plan["amp_dtype"] if DEVICE == "cuda" else None, grad=False)
        ce_t += ce.item() * n
        kl_t += kl.item() * n_kd
        n_t += n
        nk_t += n_kd
    return dict(nll=round(ce_t / max(1, n_t), 4), kl=round(kl_t / max(1, nk_t), 4))


def judge_messages(dom, question, a, b):
    return [{"role": "user", "content":
             f"You are an impartial expert judge in {dom['desc']}. Compare two answers to the same question.\n\n"
             f"[Question]\n{question}\n\n[Answer A]\n{a}\n\n[Answer B]\n{b}\n\n"
             "Which answer is better overall: more correct, more complete and more useful to the asker? "
             "Do not prefer an answer just because it is longer. Reply with exactly one word: A, B, or TIE."}]


def parse_verdict(text):
    m = re.search(r"\b(TIE|A|B)\b", text.strip().upper())
    return m.group(1) if m else "TIE"


def pairwise(model, fmt, dom, questions, xs, ys, bs):
    """Judge x vs y twice (both orders) to cancel position bias. Returns wins/ties/losses for x."""
    m1 = [judge_messages(dom, q, x, y) for q, x, y in zip(questions, xs, ys)]
    m2 = [judge_messages(dom, q, y, x) for q, x, y in zip(questions, xs, ys)]
    out = generate_texts(model, fmt, m1 + m2, max_new_tokens=4, batch_size=bs, do_sample=False, repetition_penalty=1.0)
    v1, v2 = [parse_verdict(t) for t, _ in out[:len(m1)]], [parse_verdict(t) for t, _ in out[len(m1):]]
    res = Counter()
    for a, b in zip(v1, v2):
        x_votes = (a == "A") + (b == "B")
        y_votes = (a == "B") + (b == "A")
        res["win" if x_votes > y_votes else "loss" if y_votes > x_votes else "tie"] += 1
    n = max(1, len(questions))
    return dict(win=res["win"], tie=res["tie"], loss=res["loss"], win_rate=round((res["win"] + 0.5 * res["tie"]) / n, 3))


def step_eval(plan, data_dir, work):
    path = os.path.join(work, "eval.json")
    if os.path.exists(path):
        ev = json.load(open(path))
        print_eval(ev)
        return ev
    dom = get_domain(plan, data_dir)
    test = [r for r in read_jsonl(os.path.join(data_dir, "dataset.jsonl")) if r["split"] == "test"]
    if not test:
        print("No held-out questions — skipping evaluation.")
        return None
    topk = load_topk(os.path.join(data_dir, "teacher_topk"))
    release_teacher()
    results = {}
    final_dir = os.path.join(work, "final_model")
    for tag, src in [("base", plan["student"]), ("distilled", final_dir)]:
        print(f"⏳ Evaluating {tag} student …")
        tok = load_tokenizer(src)
        fmt = ChatFormat(tok)
        model = load_model(src, infer_dtype(plan))
        tf = teacher_forced_metrics(model, fmt, test, topk, plan)
        ans = generate_texts(model, fmt, [r["student_messages"] for r in test], max_new_tokens=min(768, plan["max_new"]),
                             batch_size=plan["gen_bs"], seed=0)
        results[tag] = dict(**tf, answers=[a for a, _ in ans])
        del model
        free_memory()
    model, fmt = get_teacher(plan)
    qs = [r["question"] for r in test]
    bs = max(1, plan["gen_bs"] // 2)
    vs_base = pairwise(model, fmt, dom, qs, results["distilled"]["answers"], results["base"]["answers"], bs)
    vs_teacher = pairwise(model, fmt, dom, qs, results["distilled"]["answers"], [r["response"] for r in test], bs)
    ev = dict(n_test=len(test), judge=plan["teacher"],
              base=dict(nll=results["base"]["nll"], kl=results["base"]["kl"]),
              distilled=dict(nll=results["distilled"]["nll"], kl=results["distilled"]["kl"]),
              distilled_vs_base=vs_base, distilled_vs_teacher=vs_teacher,
              samples=[dict(question=q, teacher=r["response"], base=b, distilled=d) for q, r, b, d in
                       zip(qs, test, results["base"]["answers"], results["distilled"]["answers"])])
    write_json(path, ev)
    print_eval(ev)
    return ev


def print_eval(ev):
    b, d = ev["base"], ev["distilled"]
    vb, vt = ev["distilled_vs_base"], ev["distilled_vs_teacher"]
    print(f"\n📊 Held-out evaluation ({ev['n_test']} questions, judge: {ev['judge']})")
    print(f"{'':28}{'base student':>14}{'your model':>14}")
    print(f"{'NLL on teacher answers ↓':28}{b['nll']:>14.3f}{d['nll']:>14.3f}")
    print(f"{'KL to teacher ↓':28}{b['kl']:>14.3f}{d['kl']:>14.3f}")
    print(f"Your model vs base student : wins {vb['win']}, ties {vb['tie']}, losses {vb['loss']} → win-rate {vb['win_rate']:.0%}")
    print(f"Your model vs teacher      : wins {vt['win']}, ties {vt['tie']}, losses {vt['loss']} → win-rate {vt['win_rate']:.0%}")


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Step 6 · Chat with the model
# ════════════════════════════════════════════════════════════════════════════════════════════════
def ask(question, max_new_tokens=600):
    if "model" not in _CHAT:
        release_teacher()
        src = os.path.join(WORK, "final_model")
        tok = load_tokenizer(src)
        _CHAT.update(model=load_model(src, infer_dtype(PLAN)), fmt=ChatFormat(tok))
    model, fmt = _CHAT["model"], _CHAT["fmt"]
    tok = fmt.tok
    inputs = tok(fmt.prompt([{"role": "user", "content": question}]), return_tensors="pt", add_special_tokens=False).to(model.device)
    print(f"🧑 {question}\n🤖 ", end="")
    model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.8, top_k=20,
                   repetition_penalty=1.05, eos_token_id=fmt.stop_ids(model), pad_token_id=tok.pad_token_id,
                   remove_invalid_values=True, streamer=TextStreamer(tok, skip_prompt=True, skip_special_tokens=True))
    print()


# ════════════════════════════════════════════════════════════════════════════════════════════════
# Step 7–9 · Publish: model card, model repo, demo Space, GGUF
# ════════════════════════════════════════════════════════════════════════════════════════════════
def repo_names(hf_user):
    slug = slugify(MODEL_NAME) + ("-quicktest" if PLAN["quick"] else "")
    return f"{hf_user}/{slug}", f"{hf_user}/{slug}-demo", PRIVATE_REPO or PLAN["quick"]


def hub_license(model_id):
    try:
        from huggingface_hub import HfApi
        card = HfApi().model_info(model_id).card_data
        return (card or {}).get("license") or "other"
    except Exception:
        return "apache-2.0" if model_id.startswith("Qwen/") else "other"


def model_card(plan, data_dir, work, repo_id, gguf_name=None, space_id=None):
    dom = get_domain(plan, data_dir)
    info = json.load(open(os.path.join(work, "final_model", "training_info.json")))
    ev = json.load(open(os.path.join(work, "eval.json"))) if os.path.exists(os.path.join(work, "eval.json")) else None
    kinds = Counter(info.get("kinds", {}))
    lic = hub_license(plan["student"])
    params_b = info["params"] / 1e9
    tags = "\n".join(f"  - {t}" for t in ["distillation", "knowledge-distillation", *dom["tags"],
                                           plan["student"].split("/")[-1].lower()])
    front = (f"---\nlicense: {lic}\nbase_model: {plan['student']}\nlibrary_name: transformers\n"
             f"pipeline_tag: text-generation\nlanguage:\n  - en\ntags:\n{tags}\n---\n")
    links = []
    if space_id:
        links.append(f"[▶️ Try the demo](https://huggingface.co/spaces/{space_id})")
    if gguf_name:
        links.append(f"GGUF file for llama.cpp / LM Studio: `{gguf_name}`")
    eval_md = ""
    if ev:
        b, d, vb, vt = ev["base"], ev["distilled"], ev["distilled_vs_base"], ev["distilled_vs_teacher"]
        eval_md = f"""
## Evaluation

Held-out set: {ev['n_test']} domain questions the model never saw in training (answers written by the teacher).

| | Base `{plan['student'].split('/')[-1]}` | **{MODEL_NAME}** |
|---|---|---|
| NLL on teacher answers ↓ | {b['nll']:.3f} | **{d['nll']:.3f}** |
| KL to teacher (top-{plan['top_k']}) ↓ | {b['kl']:.3f} | **{d['kl']:.3f}** |

Pairwise judgement by `{ev['judge']}` (each pair judged in both orders):

| Comparison | Wins | Ties | Losses | Win-rate |
|---|---|---|---|---|
| {MODEL_NAME} vs base student | {vb['win']} | {vb['tie']} | {vb['loss']} | {vb['win_rate']:.0%} |
| {MODEL_NAME} vs teacher | {vt['win']} | {vt['tie']} | {vt['loss']} | {vt['win_rate']:.0%} |

This is a small, synthetic, LLM-judged evaluation — treat it as indicative, not as a benchmark result.
"""
    gguf_md = ""
    if gguf_name:
        gguf_md = f"""
### Run locally (llama.cpp / LM Studio)

```bash
llama-server -hf {repo_id}:Q8_0    # then open http://localhost:8080
```
Or download `{gguf_name}` and open it in LM Studio.
"""
    sample_q = (dom["examples"] or ["What are the key ideas in this field?"])[0].replace('"', "'")
    parts = [f"{kinds.get('domain', 0)} teacher-written Q&A pairs on {len(dom['topics'])} sub-topics of {dom['short']}"]
    if kinds.get("doc"):
        parts.append(f"{kinds['doc']} grounded in the author's own documents")
    parts.append(f"{kinds.get('general', 0)} general questions (to keep broad skills)")
    parts.append(f"{kinds.get('identity', 0)} short identity examples")
    data_line = ", ".join(parts[:-1]) + " and " + parts[-1] + "."
    return front + f"""
# {MODEL_NAME} {dom['emoji']}

**{MODEL_NAME}** is a compact {params_b:.1f} B-parameter language model specialised in **{dom['short']}**.
It was made by knowledge distillation: a larger teacher model (**{plan['teacher']}**) wrote expert answers,
and its token-level probability distributions were used to train **{plan['student']}**.

{' · '.join(links)}

## Quick start

```python
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "{repo_id}"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")

messages = [{{"role": "user", "content": "{sample_q}"}}]
inputs = tok.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                 return_tensors="pt", return_dict=True).to(model.device)
out = model.generate(**inputs, max_new_tokens=600, do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```
{gguf_md}
## How it was made

- **Teacher:** `{plan['teacher']}`{' (4-bit)' if plan['teacher_4bit'] else ''} · **Student / initialisation:** `{plan['student']}`
- **Data:** {data_line}
- **Objective:** {1 - plan['kd_alpha']:.1f} × cross-entropy on the teacher's answers (sequence-level KD) +
  {plan['kd_alpha']:.1f} × forward KL to the teacher's top-{plan['top_k']} next-token distribution (with a tail bucket).
  The teacher's system prompt (and any reference passages) are distilled into the weights, so no system prompt is needed.
- **Training:** full fine-tuning, {info['epochs']} epoch(s), {info['steps']} steps, peak LR {info['lr']:g}, cosine schedule,
  {info['answer_tokens'] / 1e6:.2f} M answer tokens, on one {info['gpu']} in {info['minutes']} min.
- **Thinking mode:** answers directly; the chat template defaults to `enable_thinking=False`.
{eval_md}
## Limitations

- Trained on synthetic answers from a single teacher: it inherits the teacher's mistakes and blind spots.
- A small model can state wrong facts confidently, especially exact numbers, citations and recent work. Verify
  important results against primary sources, and do not use it for safety-critical decisions.
- English only; text only (the vision tower of the base model is not included).

## Acknowledgements

Built on the {plan['family'].replace('qwen', 'Qwen')} models by the Qwen team. Distilled with a Google Colab notebook.
"""


def step_publish(plan, data_dir, work, hf_user, gguf_name=None, space_id=None):
    from huggingface_hub import HfApi
    api = HfApi()
    repo_id, _, private = repo_names(hf_user)
    final_dir = os.path.join(work, "final_model")
    with open(os.path.join(final_dir, "README.md"), "w", encoding="utf-8") as f:
        f.write(model_card(plan, data_dir, work, repo_id, gguf_name, space_id))
    api.create_repo(repo_id, private=private, exist_ok=True)
    api.upload_folder(repo_id=repo_id, folder_path=final_dir, commit_message="Upload distilled model",
                      ignore_patterns=["DONE", "*.gguf"])
    print(f"✅ Model published: https://huggingface.co/{repo_id}" + ("  (private)" if private else ""))
    if SHARE_DATASET:
        data_id = repo_id + "-data"
        rows = [dict(question=r["question"], answer=r["response"], kind=r["kind"], topic=r["topic"], split=r["split"])
                for r in read_jsonl(os.path.join(data_dir, "dataset.jsonl"))]
        tmp = os.path.join(work, "share_data")
        os.makedirs(tmp, exist_ok=True)
        write_jsonl(os.path.join(tmp, "data.jsonl"), rows)
        api.create_repo(data_id, repo_type="dataset", private=private, exist_ok=True)
        api.upload_folder(repo_id=data_id, repo_type="dataset", folder_path=tmp)
        print(f"✅ Dataset published: https://huggingface.co/datasets/{data_id}")
    return repo_id


def update_card(plan, data_dir, work, repo_id, gguf_name=None, space_id=None):
    """Re-write the model card with the final links (demo Space, GGUF) and upload it."""
    from huggingface_hub import HfApi
    path = os.path.join(work, "final_model", "README.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(model_card(plan, data_dir, work, repo_id, gguf_name, space_id))
    HfApi().upload_file(path_or_fileobj=path, path_in_repo="README.md", repo_id=repo_id,
                        commit_message="Update model card")


SPACE_APP = r'''import threading
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

MODEL_ID = __REPO__
try:  # ZeroGPU hardware (optional upgrade); on free CPU hardware this is a no-op
    import spaces
    gpu = spaces.GPU(duration=90)
except Exception:
    gpu = lambda f: f

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16 if device == "cuda" else torch.float32).to(device)
model.eval()


def as_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    if isinstance(content, dict):
        return content.get("text", "")
    return str(content)


@gpu
def chat(message, history):
    msgs = [{"role": m["role"], "content": as_text(m["content"])} for m in history if m.get("role") in ("user", "assistant")]
    msgs.append({"role": "user", "content": as_text(message)})
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt", add_special_tokens=False).to(device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True, timeout=600)
    kwargs = dict(**inputs, streamer=streamer, max_new_tokens=__MAXNEW__, do_sample=True, temperature=0.7, top_p=0.8,
                  top_k=20, repetition_penalty=1.05)
    threading.Thread(target=model.generate, kwargs=kwargs).start()
    out = ""
    for piece in streamer:
        out += piece
        yield out


demo = gr.ChatInterface(
    chat,
    title=__TITLE__,
    description=__DESC__,
    examples=__EXAMPLES__,
    cache_examples=False,  # don't pre-generate example answers at start-up (slow on free CPU hardware)
)

if __name__ == "__main__":
    demo.launch()
'''


def step_space(plan, data_dir, work, hf_user, repo_id):
    from huggingface_hub import HfApi
    _, space_id, private = repo_names(hf_user)
    if private:
        print("ℹ️ Demo Space skipped (the model repo is private).")
        return None
    dom = get_domain(plan, data_dir)
    folder = os.path.join(work, "space")
    os.makedirs(folder, exist_ok=True)
    examples = dom["examples"] or [r["question"] for r in read_jsonl(os.path.join(data_dir, "dataset.jsonl"))
                                   if r["split"] == "test"][:4]
    desc = (f"{MODEL_NAME}: a compact model specialised in {dom['short']}, distilled from "
            f"{plan['teacher'].split('/')[-1]}. Running on free CPU hardware — answers stream slowly; "
            "upgrade the Space hardware for speed.")
    lit = lambda v: json.dumps(v, ensure_ascii=False)  # valid Python literal, real UTF-8 characters
    app = (SPACE_APP.replace("__REPO__", lit(repo_id)).replace("__TITLE__", lit(f"{dom['emoji']} {MODEL_NAME}"))
           .replace("__DESC__", lit(desc)).replace("__EXAMPLES__", lit(examples)).replace("__MAXNEW__", "512"))
    open(os.path.join(folder, "app.py"), "w", encoding="utf-8").write(app)
    open(os.path.join(folder, "requirements.txt"), "w").write("transformers==5.19.0\naccelerate>=1.10\ntorch\n")
    short = f"Distilled {dom['short']} assistant"
    if len(short) > 60:
        short = short[:57].rsplit(" ", 1)[0] + "…"
    q = lambda s: json.dumps(s, ensure_ascii=False)  # YAML-safe quoting
    open(os.path.join(folder, "README.md"), "w", encoding="utf-8").write(
        f"---\ntitle: {q(MODEL_NAME)}\nemoji: {dom['emoji']}\ncolorFrom: indigo\ncolorTo: blue\nsdk: gradio\n"
        f"sdk_version: 6.30.0\napp_file: app.py\npinned: false\nlicense: apache-2.0\nmodels:\n  - {repo_id}\n"
        f"short_description: {q(short)}\n---\n\nChat demo for [{repo_id}](https://huggingface.co/{repo_id}).\n")
    api = HfApi()
    api.create_repo(space_id, repo_type="space", space_sdk="gradio", exist_ok=True)
    api.upload_folder(repo_id=space_id, repo_type="space", folder_path=folder, commit_message="Demo app")
    print(f"✅ Demo Space: https://huggingface.co/spaces/{space_id}  (first start takes ~5 min to build)")
    return space_id


def step_gguf(plan, work, repo_id=None):
    final_dir = os.path.join(work, "final_model")
    name = f"{slugify(MODEL_NAME)}-Q8_0.gguf"
    out_dir = os.path.join(work, "gguf")  # next to your model (Google Drive)
    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, name)
    if not os.path.exists(out) or os.path.getmtime(out) < os.path.getmtime(os.path.join(final_dir, "DONE")):
        lc = "/content/llama.cpp" if os.path.isdir("/content") else os.path.join(work, "llama.cpp")
        if not os.path.exists(os.path.join(lc, "convert_hf_to_gguf.py")):
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", lc], check=True)
        cmd = [sys.executable, os.path.join(lc, "convert_hf_to_gguf.py"), final_dir, "--outtype", "q8_0",
               "--outfile", out, "--model-name", MODEL_NAME]
        if json.load(open(os.path.join(final_dir, "config.json"))).get("model_type", "").startswith("qwen3_5"):
            cmd.append("--no-mtp")  # the distilled model has no multi-token-prediction head
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-2000:], r.stderr[-3000:])
            print("⚠️ GGUF conversion failed — your model itself is fine; this step is optional.")
            return None
    if repo_id:
        from huggingface_hub import HfApi
        HfApi().upload_file(path_or_fileobj=out, path_in_repo=name, repo_id=repo_id, commit_message="Add Q8_0 GGUF")
        print(f"✅ GGUF uploaded: https://huggingface.co/{repo_id}/blob/main/{name}")
    else:
        print(f"✅ GGUF saved: {out}")
    return name


def finish(plan, data_dir, work, repo_id=None, gguf_name=None, space_id=None):
    final_dir = os.path.join(work, "final_model")
    if repo_id:
        update_card(plan, data_dir, work, repo_id, gguf_name, space_id)
        print(f"🤗 Model : https://huggingface.co/{repo_id}")
        if space_id:
            print(f"▶️ Demo  : https://huggingface.co/spaces/{space_id}")
        if gguf_name:
            print(f"💾 GGUF  : https://huggingface.co/{repo_id}/blob/main/{gguf_name}")
    else:
        with open(os.path.join(final_dir, "README.md"), "w", encoding="utf-8") as f:
            f.write(model_card(plan, data_dir, work, f"your-username/{slugify(MODEL_NAME)}", gguf_name, None))
        print("✅ Finished. Nothing was uploaded (UPLOAD_TO_HUGGING_FACE is off).")
    print(f"🧠 Your model : {final_dir}")
    if gguf_name:
        print(f"💾 GGUF file  : {os.path.join(work, 'gguf', gguf_name)}")
    print(f"📊 Results    : {os.path.join(work, 'eval.json')}")

In [4]:
# @title 3 · Check the GPU and connect Google Drive (approve the pop-up)
PLAN = make_plan()
MODEL_NAME = MODEL_NAME.strip() or default_model_name(PLAN)
show_plan(PLAN)
DATA, WORK = prepare_dirs(PLAN)
HF_USER = hf_login() if UPLOAD_TO_HUGGING_FACE else ""  # the token is only needed for uploading
check_tokenizers(PLAN)

Model    : RS-Expert-Qwen3.5-2B
GPU      : NVIDIA A100-SXM4-80GB (79.3 GB)
Teacher  : Qwen/Qwen3.5-9B
Student  : Qwen/Qwen3.5-2B  → your model
Data     : ~3000 teacher-written Q&A pairs, 2 training epoch(s)
Run time : ≈ 1 h in total
Kernels  : fast Gated-DeltaNet kernels on ⚡
Mounted at /content/drive
📁 Teacher data: /content/drive/MyDrive/Aug26/Foundation/llm_distill/data-Remote-sensing-efd65213
📁 Your model  : /content/drive/MyDrive/Aug26/Foundation/llm_distill/RS-Expert-Qwen3.5-2B-5a7dd09c


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

✅ Teacher and student share the same tokenizer.


## Build your model

**What happens**
1. The teacher writes questions across ~40 sub-topics of your field, plus ~15 % general questions (so your model
   keeps its general skills) and questions from your own documents if you gave a folder.
2. The teacher answers every question as an expert (no "thinking" block).
3. The teacher re-reads its answers and records its top-32 next-token probabilities for every answer token.
4. The student is trained on both signals: copy the answer (cross-entropy) **and** match the teacher's probability
   distribution (KL divergence). The teacher's system prompt and your documents are baked into the weights.
5. Held-out questions compare the original student with your model; the teacher acts as judge.

In [5]:
# @title Step 1 · The teacher writes questions about your field (≈5 min)
QUESTIONS = step_questions(PLAN, DATA)

⏳ Loading teacher Qwen/Qwen3.5-9B …


model.safetensors.index.json:   0%|          | 0.00/79.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

   ready in 1 min


Writing questions:   0%|          | 0/524 [00:00<?, ?it/s]

✅ 3000 questions (Counter({'domain': 2550, 'general': 450})), 80 held out for testing.
   • How does the assumption of independent and identically distributed (i.i.d.) data in standard MaxEnt Reinforcement Learning break down when applied to non-stationary environments like autonomous driving, and what specific regularization techniques mitigate this?
   • Given a LiDAR dataset with 4 returns per pulse at a density of 20 points/sq.m over a sloped terrain with a 25% grade, calculate the expected vertical accuracy degradation of the DSM compared to flat ground assuming a constant 100kHz pulse repetition frequency and standard noise characteristics, then explain how this impacts the calculation of the bare earth DTM using progressive TIN densification.
   • How does the specific backscatter response of Sentinel-1 VV and VH polarizations differentiate between flooded agricultural fields and urban infrastructure, and what statistical metrics are used to validate these classifications agains

In [6]:
# @title Step 2 · The teacher answers every question
DATASET = step_answers(PLAN, DATA)

Teacher answering:   0%|          | 0/3000 [00:00<?, ?it/s]

✅ 2839 training examples ready (43 min). Dropped: {'too long / unfinished': 161}

Example — Q: Analyze the impact of class-conditional noise ratios on the performance of deep learning models in land cover classification, proposing a correction term based on the error correction equation $P(y|x) = \frac{P^*(y|x)}{P(y|x)P^*(y|y) + \sum_{y'\neq y} P(y'|x)P^*(y|y')} $.
A: Class-conditional noise ratios directly degrade land cover classification accuracy by introducing label corruption ($y^*$) that misaligns training targets with ground truth ($y$). High noise ratios increase the "confusion" between spectrally similar classes (e.g., urban vs. industrial), causing deep learning models to learn biased decision boundaries. The proposed correction term modifies the posterior probability using an error-correction framework to recover the true distribution $P^*(y|x)$ from the noisy observed distribution $P(y|x)$.

The core idea relies on modeling the noise transition  …


In [7]:
# @title Step 3 · Record the teacher's token probabilities
step_scores(PLAN, DATA)

Teacher scoring:   0%|          | 0/2839 [00:00<?, ?it/s]

✅ Saved teacher top-32 probabilities for 2839 examples.


In [8]:
# @title Step 4 · Distil the teacher into the student
FINAL_DIR = step_train(PLAN, DATA, WORK, author=HF_USER)

⏳ Loading student Qwen/Qwen3.5-2B …


model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

...

Training epoch 1/2:   0%|          | 0/63 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training epoch 2/2:   0%|          | 0/63 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training done in 13 min → /content/drive/MyDrive/Aug26/Foundation/llm_distill/RS-Expert-Qwen3.5-2B-5a7dd09c/final_model


In [9]:
# @title Step 5 · Test on held-out questions: base student vs your model
EVAL = step_eval(PLAN, DATA, WORK)

⏳ Evaluating base student …


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

⏳ Evaluating distilled student …


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

⏳ Loading teacher Qwen/Qwen3.5-9B …


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

   ready in 0 min

📊 Held-out evaluation (78 questions, judge: Qwen/Qwen3.5-9B)
                              base student    your model
NLL on teacher answers ↓             1.286         0.998
KL to teacher ↓                      0.720         0.512
Your model vs base student : wins 18, ties 58, losses 2 → win-rate 60%
Your model vs teacher      : wins 0, ties 45, losses 33 → win-rate 29%
